In [4]:
!pip install -q -U langchain langchain-google-genai langchain-community faiss-cpu pypdf

In [6]:
from google.colab import userdata
import os

os.environ["GOOGLE_API_KEY"] = userdata.get('GEMINI_API_KEYS')
MODEL='gemini-3.5-flash-lite'
EMB_MODEL="gemini-embedding-001"

In [14]:
from google.colab import files
uploaded=files.upload()
pdf_name=list(uploaded.keys())[0]
print(pdf_name,"Uploaded Sucessfully")

Saving sample_paper_3.pdf to sample_paper_3.pdf
sample_paper_3.pdf Uploaded Sucessfully


In [15]:
#Extract pdf data
from langchain_community.document_loaders import PyPDFLoader
docs=PyPDFLoader(pdf_name).load()
print("No of pages: ",len(docs))
print("FIRST FEW CONTENT OF PAGE 1: ",docs[0].page_content[:100])
print("META DATA OF PAGE 1: ",docs[0].metadata)

No of pages:  5
FIRST FEW CONTENT OF PAGE 1:  Notes on Rule-Based Normalisation of Historical Place
 Names
 J. H. Whitcombe
 Centre for Digital Hu
META DATA OF PAGE 1:  {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-10-04T09:05:57+00:00', 'author': 'J. H. Whitcombe', 'keywords': '', 'moddate': '2026-10-04T09:05:57+00:00', 'subject': '(unspecified)', 'title': 'Notes on Rule-Based Normalisation of Historical Place Names', 'trapped': '/False', 'source': 'sample_paper_3.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1'}


In [16]:
#chunking overlapping
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter=RecursiveCharacterTextSplitter(chunk_size=800,chunk_overlap=150)
chunks=splitter.split_documents(docs)
print(f"Chunking Done with {len(chunks)} Chunks")

Chunking Done with 28 Chunks


In [17]:
#embedding vector store faiss retreiver
from langchain_google_genai import ChatGoogleGenerativeAI,GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
embeddings=GoogleGenerativeAIEmbeddings(model=EMB_MODEL)
vectorstore=FAISS.from_documents(chunks,embeddings)
retriever=vectorstore.as_retriever(search_kwargs={"k":10})
print("EMBEDDING DONE")

EMBEDDING DONE


In [34]:
from langchain_core.prompts import (ChatPromptTemplate,MessagesPlaceholder)
from langchain_core.messages import trim_messages
from langchain_core.runnables import (RunnablePassthrough,RunnableWithMessageHistory)
from langchain_core.output_parsers import StrOutputParser
from langchain_core.chat_history import BaseChatMessageHistory,InMemoryChatMessageHistory
llm=ChatGoogleGenerativeAI(model=MODEL)
prompt=ChatPromptTemplate([
    ('system',"""
    You are a helpful question-answering assistant.

        Answer the question using ONLY the provided context.

        Use the conversation history to understand
        follow-up questions.

        If the answer is not available in the context,
        say:

        "I could not find this information in the document."
        if asked for summary summarize the papper given
        Do not make up information."""),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human","context:{context} -----------question:{question}")
])
trimmer=trim_messages(
    max_tokens=1200,
    token_counter=llm,
    include_system=True,
    allow_partial=False,
    start_on="human"
)

def format_docs(docs):
  return "\n------------\n".join(d.page_content for d in docs)

rag_chain=(
    RunnablePassthrough.assign(context=(lambda x:x["question"])| retriever | format_docs)
    |prompt
    |llm
    |StrOutputParser()
)

store={}
def get_session_history(session_id: str) -> BaseChatMessageHistory:
  if session_id not in store:
    store[session_id]=InMemoryChatMessageHistory()
  return store[session_id]

conversation_chain=RunnableWithMessageHistory(
    rag_chain,
    get_session_history,
    input_messages_key="question",
    history_messages_key="chat_history"
)

import uuid
session_id=str(uuid.uuid4())
def chat(question):
  history=get_session_history(session_id)
  trimmed_history=trimmer.invoke(history.messages)
  response=conversation_chain.invoke(
      {'question':question,"chat_history":trimmed_history},
      config={'configurable':{'session_id':session_id}}
  )
  return response

while True:
  question=input("Enter the query: ")
  if question in ['exit','bye','quit']:
    print("EXITED")
    break
  answer=chat(question)
  print(answer)



/usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py:3553: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


Enter the query: summarize me the paper


Based on the provided context, the paper is titled "Notes on Rule-Based Normalisation of Historical Place Names" by J. H. Whitcombe from the Centre for Digital Humanities. 

Summary of the paper:
* **Background & Challenge:** The paper addresses the recurring difficulty in normalising historical place names—specifically the gap between idealised settings and real data. Records are incomplete, measurements drift over time, and conventions differ between sources, meaning practical systems must tolerate noise at every stage.
* **Design & Methodology:** The authors adopt a deliberately simple design where each component can be inspected independently. Every parameter was fixed before the final evaluation to avoid tuning on held-out material. 
* **Results & Findings:** The consistent pattern is that gains are largest where the data are noisiest and smallest where simple heuristics already perform well. This matches the intuition that normalising historical place names benefits most from the